In [ ]:
import io
import time
import requests
import pandas as pd

# ---------- SETTINGS (edit these) ----------
LAT_MIN, LAT_MAX = 35.0, 36.7       # rough East TN box
LON_MIN, LON_MAX = -85.5, -82.0
START_YEAR, END_YEAR = 2021, 2025   # study window
OUT_FILE = "noaa_precip_east_tn.csv"
# -------------------------------------------

BASE = "https://www.ncei.noaa.gov/access/services/data/v1"
GHCN = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/"

# 1. Download station list + inventory
def get_text(name):
    r = requests.get(GHCN + name, timeout=120)
    r.raise_for_status()
    return io.StringIO(r.text)

st = pd.read_fwf(
    get_text("ghcnd-stations.txt"),
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=["id", "lat", "lon", "elev", "state", "name"],
)
inv = pd.read_fwf(
    get_text("ghcnd-inventory.txt"),
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 35), (36, 40), (41, 45)],
    names=["id", "lat", "lon", "element", "first_year", "last_year"],
)

# 2. Filter to TN stations in the box with PRCP covering the window
box = st[(st.state == "TN")
         & st.lat.between(LAT_MIN, LAT_MAX)
         & st.lon.between(LON_MIN, LON_MAX)]
prcp = inv[(inv.element == "PRCP")
           & (inv.first_year <= START_YEAR)
           & (inv.last_year >= END_YEAR)]
candidates = box.merge(prcp[["id", "first_year", "last_year"]], on="id")
candidates.to_csv("station_list.csv", index=False)   # keep for mapping later
print(f"{len(candidates)} stations found")

115 stations found


In [ ]:
import io
import time
import requests
import pandas as pd

# ---------- SETTINGS ----------
START_DATE = "2021-01-01"
END_DATE = "2026-01-01"
STATION_FILE = "station_list.csv"
OUT_FILE = "noaa_precip_east_tn_2021_2026.csv"
# ------------------------------

BASE = "https://www.ncei.noaa.gov/access/services/data/v1"

ids = list(pd.read_csv(STATION_FILE)["id"])
n_groups = -(-len(ids) // 10)
frames = []

for i in range(0, len(ids), 10):
    group = ids[i:i + 10]
    params = {
        "dataset": "daily-summaries",
        "stations": ",".join(group),
        "startDate": START_DATE,
        "endDate": END_DATE,
        "dataTypes": "PRCP",
        "format": "csv",
        "units": "standard",
        "includeAttributes": "true",
        "includeStationName": "true",
        "includeStationLocation": "1",
    }
    for attempt in range(3):
        r = requests.get(BASE, params=params, timeout=300)
        if r.ok and r.text.strip():
            frames.append(pd.read_csv(io.StringIO(r.text), dtype=str))
            break
        time.sleep(5 * (attempt + 1))
    else:
        print("FAILED group:", group)
    print("finished group", i // 10 + 1, "of", n_groups)
    time.sleep(1)

big = pd.concat(frames, ignore_index=True)
big.to_csv(OUT_FILE, index=False)
print("Saved", OUT_FILE, big.shape)

finished group 1 of 12
finished group 2 of 12
finished group 3 of 12
finished group 4 of 12
finished group 5 of 12
finished group 6 of 12
finished group 7 of 12
finished group 8 of 12
finished group 9 of 12
finished group 10 of 12
finished group 11 of 12
finished group 12 of 12


,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,PRCP,PRCP_ATTRIBUTES
0,US1TNAN0003,"OAK RIDGE 5.7 NE, TN US",36.019079,-84.221136,270.1,2018-01-01,0.00,",,N"
1,US1TNAN0003,"OAK RIDGE 5.7 NE, TN US",36.019079,-84.221136,270.1,2018-01-02,0.00,",,N"
2,US1TNAN0003,"OAK RIDGE 5.7 NE, TN US",36.019079,-84.221136,270.1,2018-01-03,0.00,",,N"
3,US1TNAN0003,"OAK RIDGE 5.7 NE, TN US",36.019079,-84.221136,270.1,2018-01-04,0.00,",,N"
4,US1TNAN0003,"OAK RIDGE 5.7 NE, TN US",36.019079,-84.221136,270.1,2018-01-05,0.00,",,N"
...,...,...,...,...,...,...,...,...
296726,USW00063855,"CROSSVILLE 7 NW, TN US",36.0138,-85.1345,583.1,2025-12-27,0.00,",,R"
296727,USW00063855,"CROSSVILLE 7 NW, TN US",36.0138,-85.1345,583.1,2025-12-28,0.00,",,R"
296728,USW00063855,"CROSSVILLE 7 NW, TN US",36.0138,-85.1345,583.1,2025-12-29,0.12,",,R"
296729,USW00063855,"CROSSVILLE 7 NW, TN US",36.0138,-85.1345,583.1,2025-12-30,0.00,",,R"


In [4]:
import os
big.to_csv("noaa_precip_east_tn.csv", index=False)
print("Saved to:", os.path.abspath("noaa_precip_east_tn.csv"))
print(big.shape)

Saved to: e:\COSC445FP\noaa_precip_east_tn.csv
(296731, 8)
